# Classical max-norm (a priori) error bounds for composite $Q_n$

This notebook derives and evaluates a *classical smooth* error bound (in terms of a supremum norm of derivatives)
for the composite certified estimator
$$
Q_n = \tfrac{3}{4}G_n + \tfrac{1}{4}L_{n+1}.
$$
Unlike the shape-based *a posteriori* certificate used elsewhere in the project (based on $|L_{n+1}-G_n|/4$),
the bounds here assume **smoothness** ($f\in C^{2n}$) and control the error using $\|f^{(2n)}\|_\infty$.

A standard scaling argument yields a bound of the form
$$
\bigl|\mathcal I[f]-Q_n^{\mathrm{comp}}[f]\bigr|
\le
C_n\,\frac{\|f^{(2n)}\|_\infty}{m^{2n}},
$$
for the composite rule on a uniform partition into $m$ subintervals (here we take $f(x)=\frac{1}{x}$ and specialise to $[1,2]$, so $(b-a)=1$).
In our setting one can take
$$
C_n = \frac{(n!)^4(2n-1)}{4n(2n+1)\,((2n)!)^3},
$$
so the minimal number of subintervals needed to guarantee accuracy $\varepsilon$ satisfies
$$
m \ge
\left\lceil
\Bigl(\frac{(n!)^4(2n-1)}{4n(2n+1)\,((2n)!)^2\,\varepsilon}\Bigr)^{\!1/(2n)}
\right\rceil.
$$

We illustrate this bound for several test functions and discuss additional practical sources of error
(roundoff, cancellation, and implementation pitfalls in tolerance sweeps).


In [15]:
from math import factorial, ceil
import numpy as np


In [16]:
def C_n(n: int) -> float:
    """Constant in the a priori bound for composite Q_n on an interval of length 1."""
    num = factorial(n) ** 4 * (2 * n - 1)
    den = 4 * n * (2 * n + 1) * (factorial(2 * n) ** 3)
    return num / den


def m_from_supnorm(n: int, eps: float, d2n_sup: float, length: float = 1.0) -> int:
    """Return the minimal uniform subinterval count m ensuring the max-norm bound <= eps.

    Parameters
    ----------
    n:
        Gauss order (Q_n uses G_n and L_{n+1}).
    eps:
        Target absolute error tolerance.
    d2n_sup:
        A bound on ||f^{(2n)}||_∞ on the integration interval.
    length:
        Interval length (b-a). Default is 1.0.

    Returns
    -------
    int
        Minimal m such that C_n * d2n_sup * length^{2n+1} / m^{2n} <= eps.
    """
    if eps <= 0:
        raise ValueError("eps must be positive")
    if d2n_sup < 0:
        raise ValueError("d2n_sup must be nonnegative")
    cn = C_n(n)
    # Scale: (b-a)^{2n+1} / m^{2n}
    rhs = cn * d2n_sup * (length ** (2 * n + 1)) / eps
    return int(ceil(rhs ** (1.0 / (2.0 * n))))

In [17]:
def m_inv(n: int = 2, epsilon: float = 10**(-8)) -> int:
    """A priori m bound for f(x)=1/x on [1,2].

    For f(x)=1/x, we have |f^{(k)}(x)| = k!/x^{k+1}, hence
    ||f^{(2n)}||_∞ on [1,2] equals (2n)!.
    """
    d2n_sup = factorial(2 * n)
    return m_from_supnorm(n=n, eps=epsilon, d2n_sup=d2n_sup, length=1.0)


In [18]:
print('Tolerance\t3-convex (n=2)\t5-convex (n=3)\t7-convex (n=4)\t9-convex (n=5)')
for k in range(1, 17):
    epsilon = 10**(-k)
    print(f'{epsilon: .0e}\t\t{m_inv(2, epsilon)}\t\t{m_inv(3, epsilon)}\t\t{m_inv(4, epsilon)}\t\t{m_inv(5, epsilon)}')


Tolerance	3-convex (n=2)	5-convex (n=3)	7-convex (n=4)	9-convex (n=5)
 1e-01		1		1		1		1
 1e-02		1		1		1		1
 1e-03		2		1		1		1
 1e-04		3		2		1		1
 1e-05		4		2		1		1
 1e-06		7		3		2		1
 1e-07		13		4		2		2
 1e-08		22		5		3		2
 1e-09		38		8		4		2
 1e-10		68		11		5		3
 1e-11		121		16		6		4
 1e-12		214		24		8		4
 1e-13		380		34		10		5
 1e-14		676		50		14		7
 1e-15		1202		73		18		8
 1e-16		2137		107		24		10


In [5]:
# Additional smooth test functions: exp(x) and log(1+x) on [0,1]

def m_exp(n: int, eps: float) -> int:
    """A priori m bound for f(x)=exp(x) on [0,1]."""
    # f^{(2n)} = exp(x), so ||f^{(2n)}||_∞ = e on [0,1]
    d2n_sup = float(np.e)
    return m_from_supnorm(n=n, eps=eps, d2n_sup=d2n_sup, length=1.0)


def m_log1p(n: int, eps: float) -> int:
    """A priori m bound for f(x)=log(1+x) on [0,1]."""
    # f^{(k)}(x) = (-1)^{k-1} (k-1)! / (1+x)^k for k>=1
    # hence ||f^{(2n)}||_∞ = (2n-1)! at x=0
    d2n_sup = float(factorial(2 * n - 1))
    return m_from_supnorm(n=n, eps=eps, d2n_sup=d2n_sup, length=1.0)


print('A priori m (uniform) for exp(x) on [0,1]')
print('Tolerance\t n=2\t n=3\t n=4\t n=5')
for k in range(1, 17):
    eps = 10**(-k)
    print(f'{eps: .0e}\t\t{m_exp(2, eps)}\t{m_exp(3, eps)}\t{m_exp(4, eps)}\t{m_exp(5, eps)}')

print('\nA priori m (uniform) for log(1+x) on [0,1]')
print('Tolerance\t n=2\t n=3\t n=4\t n=5')
for k in range(1, 17):
    eps = 10**(-k)
    print(f'{eps: .0e}\t\t{m_log1p(2, eps)}\t{m_log1p(3, eps)}\t{m_log1p(4, eps)}\t{m_log1p(5, eps)}')

# Other practical error types

print('\nFloating-point notes:')
eps_machine = np.finfo(float).eps
print('machine epsilon:', eps_machine)
print('A posteriori indicators like |L-G|/4 may approach roundoff for very small tolerances (~1e-15).')


A priori m (uniform) for exp(x) on [0,1]
Tolerance	 n=2	 n=3	 n=4	 n=5
 1e-01		1	1	1	1
 1e-02		1	1	1	1
 1e-03		2	1	1	1
 1e-04		3	2	1	1
 1e-05		5	2	2	1
 1e-06		9	3	2	2
 1e-07		16	4	3	2
 1e-08		28	6	3	2
 1e-09		49	9	4	3
 1e-10		87	13	5	3
 1e-11		155	19	7	4
 1e-12		275	28	9	5
 1e-13		488	40	12	6
 1e-14		868	59	16	7
 1e-15		1543	86	21	9
 1e-16		2744	127	27	11

A priori m (uniform) for log(1+x) on [0,1]
Tolerance	 n=2	 n=3	 n=4	 n=5
 1e-01		1	1	1	2
 1e-02		2	2	2	2
 1e-03		2	2	2	2
 1e-04		4	3	3	3
 1e-05		6	4	3	3
 1e-06		11	6	4	4
 1e-07		19	8	6	5
 1e-08		34	12	7	6
 1e-09		60	17	10	7
 1e-10		106	24	13	9
 1e-11		189	35	17	11
 1e-12		335	52	22	14
 1e-13		595	76	29	18
 1e-14		1058	111	39	22
 1e-15		1881	162	52	28
 1e-16		3344	238	69	35

Floating-point notes:
machine epsilon: 2.220446049250313e-16
A posteriori indicators like |L-G|/4 may approach roundoff for very small tolerances (~1e-15).


## A common implementation pitfall: tolerance sweep ordering

When producing tables for tolerances $10^{-1},\dots,10^{-16}$, it is tempting to sort tolerances
and use the previous run as a warm start (e.g. start the next search from the previous subinterval count).
If the tolerances are sorted in the *wrong* direction, the code can accidentally behave as if everything
was computed for the hardest tolerance (e.g. $10^{-16}$).

The next cell demonstrates this logic error on a toy warm-start loop.


In [6]:
tols = [10.0 ** (-k) for k in range(1, 17)]


def warm_start_demo(sorted_tols):
    n_start = 1
    out = []
    for tol in sorted_tols:
        # pretend the algorithm returns n_start updated to some increasing function of difficulty
        # (here: n_start becomes at least ceil(tol^{-0.1}))
        n_needed = max(n_start, int(np.ceil(tol ** (-0.1))))
        out.append((tol, n_needed))
        n_start = n_needed
    return out


print("Wrong: sorting from hardest to easiest (ascending tol) can freeze the table:")
demo_wrong = warm_start_demo(sorted(tols))  # smallest tol first -> hardest first
print("first few rows:", demo_wrong[:3])
print("last few rows:", demo_wrong[-3:])

print(
    "\nRight: sorting from easiest to hardest (descending tol) yields a monotone growth:"
)
demo_right = warm_start_demo(sorted(tols, reverse=True))
print("first few rows:", demo_right[:3])
print("last few rows:", demo_right[-3:])

Wrong: sorting from hardest to easiest (ascending tol) can freeze the table:
first few rows: [(1e-16, 40), (1e-15, 40), (1e-14, 40)]
last few rows: [(0.001, 40), (0.01, 40), (0.1, 40)]

Right: sorting from easiest to hardest (descending tol) yields a monotone growth:
first few rows: [(0.1, 2), (0.01, 2), (0.001, 2)]
last few rows: [(1e-14, 26), (1e-15, 32), (1e-16, 40)]


## Contrast with certified (shape-based) error bounds

The max-norm bounds above are **a priori**: they depend on $\|f^{(2n)}\|_\infty$ and can be pessimistic,
but they require only a uniform partition and no adaptivity.

In the paper's main algorithm, the bound is **a posteriori** and **shape-based**:
$$
\bigl|\mathcal I - Q_n^{\mathrm{comp}}\bigr| \le \sum_I \frac14\,\bigl|L_{n+1}^I - G_n^I\bigr|.
$$
This certificate can be much sharper in practice, and it applies to higher-order convex/concave functions
even when classical derivatives do not exist (e.g. truncated powers).

In realistic computations there is also:
- **roundoff / cancellation** when differences of close quantities are formed,
- **discretisation error** from finite precision nodes/weights,
- **implementation error** (e.g. using a fixed tolerance like $10^{-16}$ instead of $10^{-k}$ in a sweep).

The repository's unit tests and notebook outputs are meant to guard against these pitfalls.
